## Capture-24 Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [ ]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import json, sys, re
import numpy as np
import pandas as pd
from tqdm import tqdm

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import (
    resample_df,            # decimate FIR-based: (df, target_cols, factor)
    normalize_str, keyize, _keyize
)

BASE    = ROOT / "data"
RAW     = BASE / "raw_data"
CLEANED = BASE / "cleaned_premerge"
MERGED  = BASE / "merged_dataset"

# Capture-24 raw root (edit if your internal folder name differs)
RAW_CAPTURE = RAW / "Capture-24" / "capture24"

# Schema + global mapping contracts
SCHEMA_PATH       = ROOT / "Unification" / "schemas" / "continuous_stream_schema.json"
ACTIVITY_MAP_PATH = ROOT / "Unification" / "schemas" / "activity_mapping.json"

SCHEMA       = json.loads(SCHEMA_PATH.read_text())
ACT_MAP_FULL = json.loads(ACTIVITY_MAP_PATH.read_text())

UNKNOWN_ID = int(ACT_MAP_FULL.get("unknown_activity_id", 9000))
ID2NAME    = {int(x["id"]): x["name"] for x in ACT_MAP_FULL["label_set"]}

print("Paths & contracts ready.")
print("RAW_CAPTURE:", RAW_CAPTURE)
print("CLEANED    :", CLEANED)
print("Schema rate :", SCHEMA["rate_hz"])
print("Unknown ID  :", UNKNOWN_ID)


### Step 1. Ingest, preporccess and map the data 

In [ ]:
# ==========================================
# STEP 1 — Ingest Capture-24, normalize to 50Hz
# ==========================================
DICT_PATH = Path("/mnt/data/annotation-label-dictionary.csv")   # uploaded
META_PATH = Path("/mnt/data/metadata.csv")                     # uploaded (not required)

def _read_label_dictionary(dict_path: Path) -> pd.DataFrame:
    """
    Returns a dict table keyed by the exact fine-grained 'annotation' string.
    """
    d = pd.read_csv(dict_path)
    if "annotation" not in d.columns:
        raise ValueError("annotation-label-dictionary.csv must contain an 'annotation' column.")
    d["annotation_key"] = d["annotation"].astype(str)
    return d

LABEL_DICT = _read_label_dictionary(DICT_PATH)

COARSE_COL = "label:WillettsSpecific2018"
if COARSE_COL not in LABEL_DICT.columns:
    raise ValueError(f"Expected coarse label column '{COARSE_COL}' not found in label dictionary.")

ANNOTATION_TO_COARSE = dict(
    zip(LABEL_DICT["annotation_key"].astype(str), LABEL_DICT[COARSE_COL].astype(str))
)

def load_capture24_raw(
    raw_capture_dir: Path,
    downsample_to_50hz: bool = True
) -> pd.DataFrame:
    """
    Loads ALL subject .csv.gz under capture24/ as one long dataframe.
    Output columns:
      subject_id, session_id,
      timestamp_ns,
      acc_x, acc_y, acc_z,
      annotation (verbatim),
      dataset_activity_label (verbatim),
      dataset_activity_id (stable int16 from factorization of dataset_activity_label)
    """
    if not raw_capture_dir.exists():
        raise FileNotFoundError(f"RAW_CAPTURE not found: {raw_capture_dir}")

    files = sorted(list(raw_capture_dir.glob("P*.csv.gz")))
    if not files:
        raise FileNotFoundError(f"No P*.csv.gz found under: {raw_capture_dir}")

    all_rows = []
    g_const = 9.80665

    for f in tqdm(files, desc="CAPTURE-24 subjects"):
        subject_id = f.stem.replace(".csv", "")  # P001 from P001.csv.gz
        session_id = "day1"

        df = pd.read_csv(f, compression="gzip")
        df = df.rename(columns={"x": "acc_x", "y": "acc_y", "z": "acc_z"})
        df["subject_id"] = subject_id
        df["session_id"] = session_id

        # Parse datetime
        df["time_dt"] = pd.to_datetime(df["time"], errors="coerce")
        df = df.dropna(subset=["time_dt"]).sort_values("time_dt").reset_index(drop=True)

        # Exact ns since start (no float drift)
        t0 = df["time_dt"].iloc[0]
        df["timestamp_ns"] = (df["time_dt"] - t0).astype("timedelta64[ns]").astype("int64")

        # ---- Unit sanity: likely g -> m/s^2 ----
        # Heuristic: if median magnitude ~1, it's in g.
        mag_med = np.nanmedian(np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2))
        if 0.5 <= mag_med <= 2.5:
            df["acc_x"] = df["acc_x"].astype(np.float64) * g_const
            df["acc_y"] = df["acc_y"].astype(np.float64) * g_const
            df["acc_z"] = df["acc_z"].astype(np.float64) * g_const

        # Native dataset label = verbatim raw annotation
        df["dataset_activity_label"] = df["annotation"].astype("string")

        # Downsample 100 Hz -> 50 Hz
        if downsample_to_50hz:
            target_cols = ["acc_x", "acc_y", "acc_z"]
            df = resample_df(df, target_cols=target_cols, factor=2)

            # IMPORTANT: enforce strict 50Hz time grid post-decimation
            # 50Hz => 20ms => 20_000_000 ns
            n = len(df)
            df["timestamp_ns"] = (np.arange(n, dtype=np.int64) * 20_000_000)

        # Keep minimal (NOTE: keep timestamp_ns, not timestamp_s)
        df = df[[
            "subject_id", "session_id",
            "timestamp_ns",
            "acc_x", "acc_y", "acc_z",
            "annotation",
            "dataset_activity_label"
        ]]

        all_rows.append(df)

    raw = pd.concat(all_rows, ignore_index=True)

    # Stable dataset_activity_id from factorization of dataset_activity_label
    codes, _ = pd.factorize(raw["dataset_activity_label"].astype(str), sort=True)
    raw["dataset_activity_id"] = (codes + 1).astype(np.int16)  # 1..K

    # Quick summary
    print("\n=== RAW SUMMARY (CAPTURE-24 wrist) ===")
    print(f"Files: {len(files)} | Rows: {raw.shape[0]:,} | Cols: {raw.shape[1]}")
    top = raw["annotation"].value_counts().head(10)
    print("\nTop annotations (raw):")
    for lbl, cnt in top.items():
        print(f"  {str(lbl)[:60]:60s} {cnt:,}")

    return raw

raw_capture24 = load_capture24_raw(RAW_CAPTURE, downsample_to_50hz=True)
raw_capture24.head(3)


### Step 2. Map the data and audit the mapping

In [ ]:
# ============================================
# STEP 2 — Map + audit: annotation -> coarse -> global_id (via activity_mapping.json)
# ============================================
if raw_capture24.empty:
    raise SystemExit("No CAPTURE-24 rows after loading. Check RAW_CAPTURE path/layout.")

MAPPING = ACT_MAP_FULL["mapping"]  # str -> global id

# annotation -> coarse label (from dictionary); fallback = "unknown"
raw_capture24["coarse_label"] = raw_capture24["annotation"].astype(str).map(
    lambda a: ANNOTATION_TO_COARSE.get(a, "unknown")
).astype("string")

def coarse_to_global_id(coarse: str) -> int:
    # normalize to match keys in ACT_MAP_FULL["mapping"]
    # use whatever your repo uses consistently; normalize_str is the safest baseline
    k = normalize_str(str(coarse))
    return int(MAPPING.get(k, UNKNOWN_ID))

raw_capture24["global_activity_id"] = raw_capture24["coarse_label"].map(coarse_to_global_id).astype("int16")
raw_capture24["global_activity_label"] = raw_capture24["global_activity_id"].map(
    lambda x: ID2NAME.get(int(x), "other")
).astype("string")

# Audit coverage
cov = (raw_capture24["global_activity_id"] != UNKNOWN_ID).mean() * 100
print(f"Global mapping coverage: {cov:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop coarse labels:")
print(raw_capture24["coarse_label"].value_counts().head(15))

unmapped = raw_capture24.loc[raw_capture24["global_activity_id"] == UNKNOWN_ID, "coarse_label"].value_counts().head(15)
print("\nUnmapped coarse labels (top):")
print(unmapped)


### Step 3. Build and clean dataset in stream json fromat

In [ ]:
# =========================================================
# STEP 3 — Build schema-ordered continuous_stream (v3) df
# =========================================================
def to_continuous_stream_capture24(df_raw: pd.DataFrame, dataset_name: str = "capture24") -> pd.DataFrame:
    if df_raw.empty:
        return pd.DataFrame(columns=[c["name"] for c in SCHEMA["columns"]])

    out = pd.DataFrame({
        "dataset":              dataset_name,
        "subject_id":           df_raw["subject_id"].astype("string"),
        "session_id":           df_raw["session_id"].astype("string"),
        "timestamp_ns":         df_raw["timestamp_ns"].astype("int64"),

        "acc_x": df_raw["acc_x"].astype("float32"),
        "acc_y": df_raw["acc_y"].astype("float32"),
        "acc_z": df_raw["acc_z"].astype("float32"),

        # No gyro in CAPTURE-24
        "gyro_x": pd.Series([pd.NA] * len(df_raw), dtype="Float32"),
        "gyro_y": pd.Series([pd.NA] * len(df_raw), dtype="Float32"),
        "gyro_z": pd.Series([pd.NA] * len(df_raw), dtype="Float32"),

        "global_activity_id":    df_raw["global_activity_id"].astype("int16"),
        "global_activity_label": df_raw["global_activity_label"].astype("string"),

        "dataset_activity_id":   df_raw["dataset_activity_id"].astype("int16"),
        "dataset_activity_label":df_raw["dataset_activity_label"].astype("string"),
    })

    order = [c["name"] for c in SCHEMA["columns"]]
    return out[order]

capture24_df = to_continuous_stream_capture24(raw_capture24, dataset_name="capture24")
print("UNIFIED rows:", len(capture24_df))
capture24_df.head(3)


### Step 4. Audit check the unified frame

In [ ]:
# ==========================================
# STEP 4 — Contract checks & quick QA
# ==========================================
print("Subjects:", capture24_df["subject_id"].nunique(),
      "| Sessions:", capture24_df["session_id"].nunique())

# Monotonic timestamp per (subject, session)
viol = 0
for (_sid, _sess), g in capture24_df.groupby(["subject_id","session_id"], sort=False):
    ts = g["timestamp_ns"].to_numpy()
    if ts.size and not np.all(np.diff(ts) >= 0):
        viol += 1
print("Monotonic violations (groups):", viol)

# Approx Hz from ns timestamps
def est_hz_ns(ts_ns: pd.Series):
    arr = ts_ns.to_numpy()
    if arr.size < 3: return np.nan
    dt = np.diff(arr) / 1e9
    dt = dt[(dt > 0) & np.isfinite(dt)]
    return float(np.median(1.0 / dt)) if dt.size else np.nan

hz = capture24_df.groupby(["subject_id","session_id"])["timestamp_ns"].apply(est_hz_ns)
print(f"Median Hz: {np.nanmedian(hz.values):.2f} (target={SCHEMA['rate_hz']})")

# Required-not-null coverage
req = SCHEMA["expectations"]["required_not_null"]
pct = capture24_df[req].notnull().all(axis=1).mean() * 100
print(f"Rows meeting required-not-null: {pct:.2f}%")

# Mapping coverage
cov = (capture24_df["global_activity_id"] != UNKNOWN_ID).mean() * 100
print(f"Global mapping coverage: {cov:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-15 canonical labels:")
print(capture24_df["global_activity_label"].value_counts().head(15))


### Step 5. Save outputs

In [ ]:
# =========================
# STEP 5 — Save outputs
# =========================
CLEANED.mkdir(parents=True, exist_ok=True)
out_path = CLEANED / "capture24.parquet"

capture24_df.to_parquet(out_path, index=False)
print("Wrote:", out_path)
